# 04 · Advanced Methods: MLP + Gradient-Based Attribution
Trains a PyTorch MLP (MPS-accelerated on Apple Silicon), applies Integrated
Gradients, SmoothGrad, DeepLIFT, GradientSHAP, and LIME, then produces a
grand cross-method comparison matrix.

**Input:** `data.pkl`, `models.pkl`

## Extension hooks
- Add receptor pairs: load `data.pkl` with the new pair from `01_data.ipynb`
- Add methods: follow the pattern in Section 4
- Add model architectures: replace MolMLP with a GNN or Transformer

In [ ]:
import sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from utils import (build_feature_matrix, expand_shap_to_fp, SEED, N_FP)
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── Captum / numpy 2.x compatibility fix ─────────────────────────────────────
# captum < 0.7 internally calls np.bool/np.int (removed in numpy 2.0)
# Fix: either upgrade captum or patch the numpy shim before importing
import numpy as np
if tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0):
    import sys
    # Patch removed numpy aliases back in (safe shim for captum internals)
    np.bool   = bool    # type: ignore
    np.int    = int     # type: ignore
    np.float  = float   # type: ignore
    np.complex= complex # type: ignore
    np.object = object  # type: ignore
    np.str    = str     # type: ignore

try:
    from captum.attr import IntegratedGradients, NoiseTunnel, DeepLift, GradientShap
    HAS_CAPTUM = True; print("captum OK")
except ImportError:
    HAS_CAPTUM = False; print("captum not found — pip install captum")
except RuntimeError as e:
    print(f"captum runtime error: {e}")
    print("Try: pip install captum --upgrade  OR  pip install 'numpy<2.0'")
    HAS_CAPTUM = False

try:
    import lime.lime_tabular
    HAS_LIME = True; print("lime OK")
except ImportError:
    HAS_LIME = False; print("lime not found — pip install lime")

import shap

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# MPS device (Apple Silicon)
def get_device():
    if torch.backends.mps.is_available(): return torch.device('mps')
    if torch.cuda.is_available():         return torch.device('cuda')
    return torch.device('cpu')
DEVICE = get_device()
print(f"Device: {DEVICE}")

## 1. Load data & models

In [ ]:
import json as _j, os as _os, tempfile as _tmp

with open('data.pkl','rb')  as f: data   = pickle.load(f)
with open('models.pkl','rb') as f: _saved = pickle.load(f)

d2 = data['d2']; sht = data['sht']; merged = data['merged']
X_d2 = data['X_d2']; X_sht = data['X_sht']; X_ov = data['X_ov']
sel = _saved['selectors']; sc = _saved['scalers']

y_d2  = d2['pChEMBL'].values
y_sht = sht['pChEMBL'].values
y_del = merged['delta'].values
smiles_ov = merged['curated_smiles'].tolist()

# ── Extract models — handle both pkl formats ──────────────────────────────────
def _patch_xgb(model):
    if not hasattr(model, 'save_model'): return model
    t = _tmp.NamedTemporaryFile(suffix='.json', delete=False); t.close()
    try:
        model.save_model(t.name)
        with open(t.name) as f: mj = _j.load(f)
        bs = mj['learner']['learner_model_param']['base_score']
        if isinstance(bs, str) and bs.startswith('['):
            mj['learner']['learner_model_param']['base_score'] = bs[1:-1]
            with open(t.name, 'w') as f: _j.dump(mj, f)
            model.load_model(t.name)
    except Exception: pass
    finally: _os.unlink(t.name)
    return model

if 'best_models' in _saved:           # old format (model_tuning.ipynb)
    _bm      = _saved['best_models']
    rf_d2    = _patch_xgb(_bm['D2']['model'])
    rf_5ht2a = _patch_xgb(_bm['5HT2A']['model'])
    rf_sel   = _patch_xgb(_bm['Delta']['model'])
    print(f"Loaded (old format): D2={_bm['D2']['model_name']} R2={_bm['D2']['test_r2']:.3f} | "
          f"5HT2A={_bm['5HT2A']['model_name']} R2={_bm['5HT2A']['test_r2']:.3f} | "
          f"Delta={_bm['Delta']['model_name']} R2={_bm['Delta']['test_r2']:.3f}")
else:                                  # new format (02_models.ipynb)
    rf_d2    = _patch_xgb(_saved['rf_d2']['model'])
    rf_5ht2a = _patch_xgb(_saved['rf_5ht2a']['model'])
    rf_sel   = _patch_xgb(_saved['rf_sel']['model'])
    print(f"Loaded (new format): D2 R2={_saved['rf_d2']['test_r2']:.3f} | "
          f"5HT2A R2={_saved['rf_5ht2a']['test_r2']:.3f} | "
          f"Delta R2={_saved['rf_sel']['test_r2']:.3f}")

# ── Rebuild feature matrices ──────────────────────────────────────────────────
X_ov_feat = build_feature_matrix(X_ov, smiles_ov, sel['overlap'], sc['overlap'])
X_d2_ov   = build_feature_matrix(X_ov, smiles_ov, sel['D2'],      sc['D2'])
X_sht_ov  = build_feature_matrix(X_ov, smiles_ov, sel['5HT2A'],   sc['5HT2A'])

y_d2_ov  = merged['pChEMBL_D2'].values
y_sht_ov = merged['pChEMBL_5HT2A'].values

N_FEAT = X_ov_feat.shape[1]
print(f"Feature dims: overlap={X_ov_feat.shape[1]}  D2={X_d2_ov.shape[1]}  5HT2A={X_sht_ov.shape[1]}")
print(f"rf_sel type: {type(rf_sel).__name__}")

## 2. MLP architecture & training

In [ ]:
class MolMLP(nn.Module):
    def __init__(self, input_dim, hidden=[1024,512,256], dropout=0.3):
        super().__init__()
        layers, in_d = [], input_dim
        for h in hidden:
            layers += [nn.Linear(in_d,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers.append(nn.Linear(in_d, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)

from utils import scaffold_split as _scaffold_split

def train_mlp(X, y, df_for_split, label, epochs=150, batch=256, lr=1e-3, patience=20):
    """Train MLP with scaffold-stratified train/test split."""
    tr_idx, te_idx = _scaffold_split(df_for_split, test_frac=0.2, seed=SEED)
    Xtr, Xte = X[tr_idx], X[te_idx]
    ytr, yte  = y[tr_idx], y[te_idx]

    t = lambda a: torch.tensor(a, dtype=torch.float32).to(DEVICE)
    loader  = DataLoader(TensorDataset(t(Xtr), t(ytr)), batch_size=batch, shuffle=True)
    model   = MolMLP(X.shape[1]).to(DEVICE)
    opt     = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched   = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
    loss_fn = nn.MSELoss()

    best_r2, best_state, wait = -np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            yp = model(t(Xte)).cpu().numpy()
        r2 = r2_score(yte, yp); sched.step(-r2)
        if r2 > best_r2:
            best_r2  = r2
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        yp = model(t(Xte)).cpu().numpy()
    r2   = r2_score(yte, yp)
    rmse = np.sqrt(mean_squared_error(yte, yp))
    mae  = mean_absolute_error(yte, yp)
    print(f"  {label:35s} | N_tr={len(tr_idx):4d} N_te={len(te_idx):4d} "
          f"| R2={r2:.3f} | RMSE={rmse:.3f} | MAE={mae:.3f}  [scaffold split]")
    return model

# ── Feature matrices on overlap set ──────────────────────────────────────────
# (already built in load cell — just reuse)
y_d2_ov  = merged['pChEMBL_D2'].values
y_sht_ov = merged['pChEMBL_5HT2A'].values

print("Training MLPs with scaffold split...")
mlp_d2    = train_mlp(X_d2_ov,   y_d2_ov,  merged, 'mlp_d2    (D2,    overlap, scaffold)')
mlp_5ht2a = train_mlp(X_sht_ov,  y_sht_ov, merged, 'mlp_5ht2a (5HT2A, overlap, scaffold)')
mlp_sel   = train_mlp(X_ov_feat,  y_del,    merged, 'mlp_sel   (Delta, overlap, scaffold)')

## 3. Integrated Gradients + SmoothGrad + DeepLIFT + GradientSHAP

In [ ]:
assert HAS_CAPTUM, "Install captum: pip install captum"

N_SUB = 500
idx_sub = np.random.choice(len(X_ov), N_SUB, replace=False)

X_d2_sub  = torch.tensor(X_d2_ov[idx_sub],  dtype=torch.float32).to(DEVICE)
X_sht_sub = torch.tensor(X_sht_ov[idx_sub], dtype=torch.float32).to(DEVICE)
X_ov_sub  = torch.tensor(X_ov_feat[idx_sub],dtype=torch.float32).to(DEVICE)
zero_d2   = torch.zeros(1, X_d2_ov.shape[1],  dtype=torch.float32).to(DEVICE)
zero_sht  = torch.zeros(1, X_sht_ov.shape[1], dtype=torch.float32).to(DEVICE)
zero_ov   = torch.zeros(1, X_ov_feat.shape[1], dtype=torch.float32).to(DEVICE)

def batch_attr(explainer, X_t, baseline, batch=64, **kw):
    """
    Handles three captum baseline conventions:
    - Single reference (shape 1×F): expand to match batch size  [IG, DeepLift, SmoothGrad]
    - Background dataset (shape N×F): pass as-is               [GradientShap]
    - NoiseTunnel wrapper: baselines must be a keyword arg
    """
    out = []
    is_nt          = isinstance(explainer, NoiseTunnel)
    single_baseline = baseline.shape[0] == 1   # True for zero-vectors, False for bg datasets

    for s in range(0, len(X_t), batch):
        xb = X_t[s:s+batch]
        bb = baseline.expand(len(xb), -1) if single_baseline else baseline

        try:
            if is_nt:
                a = explainer.attribute(xb, baselines=bb, **kw)
            else:
                a = explainer.attribute(xb, bb, **kw)
        except (RuntimeError, NotImplementedError):
            # MPS fallback
            try:
                inner_fn = explainer.attribution_method.forward_func
            except AttributeError:
                inner_fn = getattr(explainer, 'model',
                           getattr(explainer, 'forward_func', None))
            if inner_fn is not None:
                inner_fn.to('cpu')
            if is_nt:
                a = explainer.attribute(xb.cpu(), baselines=bb.cpu(), **kw)
            else:
                a = explainer.attribute(xb.cpu(), bb.cpu(), **kw)
            if inner_fn is not None:
                inner_fn.to(DEVICE)

        out.append(a.detach().cpu().numpy())
    return np.vstack(out)

# Integrated Gradients (zero baseline, single reference)
print("IG...")
ig_d2    = batch_attr(IntegratedGradients(mlp_d2),    X_d2_sub,  zero_d2,  n_steps=50)
ig_5ht2a = batch_attr(IntegratedGradients(mlp_5ht2a), X_sht_sub, zero_sht, n_steps=50)
ig_sel   = batch_attr(IntegratedGradients(mlp_sel),   X_ov_sub,  zero_ov,  n_steps=50)

# SmoothGrad — NoiseTunnel around IG, zero baseline
print("SmoothGrad...")
sg_sel = batch_attr(
    NoiseTunnel(IntegratedGradients(mlp_sel)),
    X_ov_sub, zero_ov,
    nt_type='smoothgrad', nt_samples=10, stdevs=0.05, n_steps=25
)

# DeepLIFT (zero baseline)
print("DeepLIFT...")
dl_sel = batch_attr(DeepLift(mlp_sel), X_ov_sub, zero_ov)

# GradientSHAP — background dataset as baseline (NOT a single reference)
print("GradientSHAP...")
bg_t   = X_ov_sub[:min(100, len(X_ov_sub))]   # shape (100, F) — background distribution
gs_sel = batch_attr(GradientShap(mlp_sel), X_ov_sub, bg_t, n_samples=5)

print("All gradient attributions computed.")

## 4. IG: A vs B comparison

In [ ]:
# Expand IG attributions to common 2048-bit space
ig_A = expand_shap_to_fp(ig_sel,    sel['overlap'])
ig_B = expand_shap_to_fp(ig_5ht2a,  sel['5HT2A']) - expand_shap_to_fp(ig_d2, sel['D2'])

imp_igA = np.abs(ig_A).mean(axis=0)
imp_igB = np.abs(ig_B).mean(axis=0)

from scipy.stats import spearmanr, pearsonr
rho_ig, _  = spearmanr(imp_igA, imp_igB)
pear_ig, _ = pearsonr(imp_igA,  imp_igB)
cos_ig     = cosine_similarity(ig_A, ig_B).diagonal()
ol_ig      = len(set(np.argsort(imp_igA)[::-1][:50].tolist()) &
                  set(np.argsort(imp_igB)[::-1][:50].tolist()))

print(f"IG  A vs B: rho={rho_ig:.4f}  pearson={pear_ig:.4f}  "
      f"top50={ol_ig}/50  cos={cos_ig.mean():.4f}")

## 5. Grand cross-method comparison

In [ ]:
# All Approach-A importance vectors in common FP space
method_A = {
    'SHAP-RF': np.abs(expand_shap_to_fp(
        shap.TreeExplainer(rf_sel,
            feature_perturbation='tree_path_dependent'
        ).shap_values(build_feature_matrix(
            X_ov[idx_sub], [smiles_ov[i] for i in idx_sub],
            sel['overlap'], sc['overlap'])),
        sel['overlap'])).mean(axis=0),
    'IG':          imp_igA,
    'SmoothGrad':  np.abs(expand_shap_to_fp(sg_sel, sel['overlap'])).mean(axis=0),
    'DeepLIFT':    np.abs(expand_shap_to_fp(dl_sel, sel['overlap'])).mean(axis=0),
    'GradSHAP':    np.abs(expand_shap_to_fp(gs_sel, sel['overlap'])).mean(axis=0),
}

names = list(method_A.keys())
n     = len(names)
rho_m = np.ones((n, n))
for i in range(n):
    for j in range(i+1, n):
        r, _ = spearmanr(list(method_A.values())[i],
                          list(method_A.values())[j])
        rho_m[i,j] = rho_m[j,i] = r

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(rho_m, vmin=0, vmax=1, cmap='YlGn', aspect='auto')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(names, fontsize=10)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{rho_m[i,j]:.2f}', ha='center', va='center',
                fontsize=11, color='black' if rho_m[i,j] < 0.85 else 'white')
plt.colorbar(im, ax=ax, label='Spearman rho')
ax.set_title('Grand cross-method Spearman matrix (Approach A — selectivity model)',
             fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig_04_grand_matrix.png', dpi=130, bbox_inches='tight')
plt.show()

print("\nConclusion: methods that show rho > 0.7 with each other recover the same signal.")
print("Low rho between two methods = they make different assumptions about feature relevance.")

---
## Extension hooks

```python
# ── Add a new receptor pair ───────────────────────────────────────────────────
# 1. In 01_data.ipynb, add the pair to TARGETS dict
# 2. Run 01_data.ipynb and 02_models.ipynb
# 3. Load the new pair here and repeat the IG / SHAP pipeline

# ── Add LIME ──────────────────────────────────────────────────────────────────
# See atypicality_gradients.ipynb → Part 4 for the LIME implementation
# Drop it in here as Section 6

# ── Add GNN model ─────────────────────────────────────────────────────────────
# Replace mlp_sel with a Chemprop D-MPNN trained on SMILES
# Use GNNExplainer or SubgraphX for Approach A attribution
# Expand node-level attributions to the common bit space via
# ECFP bit-to-atom mapping for cross-method comparison
```